In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
def limpiar_open_meteo():
    ruta_raw = "../data/raw/open-meteo.csv"

    df = pd.read_csv(ruta_raw)
    df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.to_period("M")
    

    df = df.groupby("time").agg({
        "relative_humidity_2m (%)" : ["mean"],
        "surface_pressure (hPa)" : ["max", "min", "mean"],
        "wind_speed_100m (km/h)" : ["max", "mean"]
    })

    return df


In [4]:
def limpiar_copernicus():
    ruta_raw = "../data/raw/valencia_mar_temperatura.csv"

    df = pd.read_csv(ruta_raw)
    df["time"] = pd.to_datetime(df["time"], format= "%Y-%m-%d", errors= "coerce").dt.to_period("M")
    df = df.set_index("time")

    df_temp_agua = np.round(df.loc[:,["thetao"]].copy(), decimals=1)
    print(df_temp_agua)
    return df_temp_agua


In [5]:
def limpiar_max_min_columns(df:pd.DataFrame, column:str):
    df.loc[:,column] = df[column].str.partition("(")[0]
    df[column] = df[column].astype(float)

def limpiar_aemet():
    ruta_raw = "../data/raw/datos_climatologicos_valencia.json"

    df = pd.read_json(ruta_raw)
    # limpiar la fecha
    df["fecha"] = pd.to_datetime(df["fecha"], format= "%Y-%m", errors= "coerce").dt.to_period("M")
    df = df.dropna(subset=["fecha"])
    df = df.set_index("fecha")
    df = df.sort_index(ascending= True)
    # limpiar ta_max
    limpiar_max_min_columns(df, "ta_max")
    # limpiar ta_min
    limpiar_max_min_columns(df, "ta_min")
    # limpiar p_max
    limpiar_max_min_columns(df, "p_max")
    # # limpiar np_100
    # df["np_100"] = df["np_100"].astype("Int64")
    # # limpiar np_300
    # df["np_300"] = df["np_300"].astype("Int64")
    # limpiar w_racha
    df.loc[:, "w_racha"] = df["w_racha"].str.split(r'[/(]', regex=True).str[1]
    df["w_racha"] = df["w_racha"].astype(float)
    df["w_racha"] = df["w_racha"] * 3.6 # Pasar de m/s a km/h
    # limpiar q_max
    limpiar_max_min_columns(df, "q_max")
    # limpiar q_min
    limpiar_max_min_columns(df, "q_min")

    df_aemet =  df.loc[:,["tm_mes", "tm_max", "tm_min", "ta_max", "ta_min", "nt_30", "p_mes", "p_max", "np_100", "np_300", 
                          "n_llu", "n_gra", "n_tor"]].copy()
    return df_aemet

In [9]:
import pandas as pd
import os

def limpiar_max_min_columns_diario(df: pd.DataFrame, column: str):
    if column in df.columns:
        # 1. Extraemos, convertimos a texto plano, limpiamos paréntesis y comas
        serie_limpia = df[column].astype(str).str.partition("(")[0].str.replace(',', '.', regex=False).str.strip()
        # 2. Convertimos directamente a numérico (los errores o vacíos pasan a NaN)
        df[column] = pd.to_numeric(serie_limpia, errors='coerce')

def limpiar_aemet_diario():
    ruta_raw = "../data/raw/datos_climatologicos_valencia_diarios_1991_2025.json"

    df = pd.read_json(ruta_raw)
    
    # 1. Limpiar la fecha en formato diario real (YYYY-MM-DD)
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")
    df = df.dropna(subset=["fecha"])
    df = df.set_index("fecha")
    df.index.name = "fecha"
    df = df.sort_index(ascending=True)
    
    # 2. Columnas numéricas del JSON diario de AEMET que queremos limpiar
    columnas_diarias = ["tmed", "tmin", "tmax", "prec", "velmedia", "racha", "sol", "presMax", "presMin", "hrMedia"]
    
    for col in columnas_diarias:
        limpiar_max_min_columns_diario(df, col)

    # 3. Filtrar solo las columnas que existan realmente en el DataFrame
    columnas_disponibles = [col for col in columnas_diarias if col in df.columns]
    df_aemet = df.loc[:, columnas_disponibles].copy()

    # 4. Asegurar que la carpeta de destino existe y guardar en parquet
    os.makedirs("../data/processed", exist_ok=True)
    ruta_parquet = "../data/processed/datos_meteo_Valencia_91-25(days).parquet"
    
    df_aemet.to_parquet(ruta_parquet)
    print(f"¡Fichero diario guardado con éxito en: {ruta_parquet}!")
    print(df_aemet.info())
    
    return df_aemet

# Ejecutar la función
df_aemet = limpiar_aemet_diario()

¡Fichero diario guardado con éxito en: ../data/processed/datos_meteo_Valencia_91-25(days).parquet!
<class 'pandas.DataFrame'>
DatetimeIndex: 12784 entries, 1991-01-01 to 2025-12-31
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   tmed      12784 non-null  float64
 1   tmin      12784 non-null  float64
 2   tmax      12784 non-null  float64
 3   prec      12260 non-null  float64
 4   velmedia  7300 non-null   float64
 5   racha     7279 non-null   float64
 6   sol       12356 non-null  float64
 7   presMax   7310 non-null   float64
 8   presMin   7310 non-null   float64
 9   hrMedia   7307 non-null   float64
dtypes: float64(10)
memory usage: 1.1 MB
None


In [15]:
df_aemet = limpiar_aemet()
print(df_aemet.info())
df_copernicus = limpiar_copernicus()
print(df_copernicus.info())
df_open_meteo = limpiar_open_meteo()
print(df_open_meteo.info())

<class 'pandas.DataFrame'>
PeriodIndex: 420 entries, 1991-01 to 2025-12
Freq: M
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tm_mes  420 non-null    float64
 1   tm_max  420 non-null    float64
 2   tm_min  420 non-null    float64
 3   ta_max  420 non-null    float64
 4   ta_min  420 non-null    float64
 5   nt_30   420 non-null    int64  
 6   p_mes   420 non-null    float64
 7   p_max   419 non-null    float64
 8   np_100  418 non-null    float64
 9   np_300  418 non-null    float64
 10  n_llu   420 non-null    int64  
 11  n_gra   420 non-null    int64  
 12  n_tor   420 non-null    int64  
dtypes: float64(9), int64(4)
memory usage: 45.9 KB
None
         thetao
time           
1991-01    13.6
1991-02    12.5
1991-03    13.4
1991-04    14.5
1991-05    16.5
...         ...
2025-08    27.9
2025-09    25.8
2025-10    22.9
2025-11    19.4
2025-12    16.3

[420 rows x 1 columns]
<class 'pandas.DataFrame'>
PeriodIndex: 420 

In [16]:
filas_con_nulos = df_aemet[df_aemet.isna().any(axis=1)]

# # Mostrar todas las filas (cuidado si tu DataFrame tiene miles de filas, ocupará toda la página)
# pd.set_option('display.max_rows', None)

# # Mostrar todas las columnas (ideal para que no se oculten variables climáticas)
# pd.set_option('display.max_columns', None)

# # (Opcional) Evitar que se corten los textos largos dentro de las celdas
# pd.set_option('display.max_colwidth', None)

# pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')

filas_con_nulos

,tm_mes,tm_max,tm_min,ta_max,ta_min,nt_30,p_mes,p_max,np_100,np_300,n_llu,n_gra,n_tor
fecha,,,,,,,,,,,,,
2005-09,23.1,27.6,18.6,37.5,14.0,5,49.0,26.2,NaN,NaN,8,0,0
2006-01,10.5,14.4,6.6,20.1,-0.6,0,66.6,NaN,NaN,NaN,7,0,0


In [17]:
len(filas_con_nulos)

2

In [18]:
for i in range(len(filas_con_nulos)):
    # Obtenemos la etiqueta de la fecha actual y sus valores
    fecha_actual = filas_con_nulos.index[i]
    p_val = filas_con_nulos.iloc[i]["p_mes"]
    n_val = filas_con_nulos.iloc[i]["n_llu"]
    
    # 1. Filtramos excluyendo únicamente la fecha de esta iteración
    df_fill1 = df_aemet[
        (df_aemet.index != fecha_actual) &
        (df_aemet["p_mes"].between(p_val * 0.9, p_val * 1.1)) &
        (df_aemet["n_llu"].between(n_val * 0.9, n_val * 1.1))
    ]

    # 2. Asignamos de forma segura usando .loc con la etiqueta real (fecha_actual)
    if pd.isna(filas_con_nulos.loc[fecha_actual, "np_100"]):
        df_aemet.loc[fecha_actual, "np_100"] = df_fill1["np_100"].mean()
        
    if pd.isna(filas_con_nulos.loc[fecha_actual, "np_300"]):
        df_aemet.loc[fecha_actual, "np_300"] = df_fill1["np_300"].mean()
        
    if pd.isna(filas_con_nulos.loc[fecha_actual, "p_max"]):
        df_aemet.loc[fecha_actual, "p_max"] = df_fill1["p_max"].mean()



In [19]:
df_open_meteo.columns = ['_'.join(col).strip() for col in df_open_meteo.columns.values]

df = pd.concat([df_aemet, df_copernicus, df_open_meteo], axis=1)

df = df.rename(columns={
    "tm_mes" : "temp_med_mes (Cº)", 
    "tm_max" : "temp_med_max (Cº)", 
    "tm_min" : "temp_med_min (Cº)", 
    "ta_max" : "temp_abs_max (Cº)", 
    "ta_min" : "temp_abs_min (Cº)", 
    "nt_30" : "dias_temp>30", 
    "p_mes" : "total_precip_mes (mm)", 
    "p_max" : "precip_max (mm)", 
    "np_100" : "dias_precip>10mm",
    "np_300" : "dias_precip>30mm", 
    "n_llu" : "dias_lluvia", 
    "n_gra" : "dias_granizo", 
    "n_tor" : "dias_tormenta", 
    "thetao" : "temp_med_mar (Cº)",
    # Nombres planos resultantes de Open-Meteo tras el aplanado:
    "relative_humidity_2m (%)_mean" : "humedad_med (%)", 
    "surface_pressure (hPa)_max" : "presion_max (hPa)", 
    "surface_pressure (hPa)_min" : "presion_min (hPa)",
    "surface_pressure (hPa)_mean" : "presion_med (hPa)", 
    "wind_speed_100m (km/h)_max" : "viento_vel_max (km/h)",   
    "wind_speed_100m (km/h)_mean" : "viento_vel_med (km/h)"
})
df = df[[
    "temp_med_mar (Cº)", "temp_med_mes (Cº)", "temp_med_max (Cº)", "temp_med_min (Cº)", 
    "temp_abs_max (Cº)", "temp_abs_min (Cº)", "dias_temp>30", "total_precip_mes (mm)", 
    "precip_max (mm)", "dias_precip>10mm", "dias_precip>30mm", "dias_lluvia", 
    "dias_granizo", "dias_tormenta", "humedad_med (%)", "presion_max (hPa)", 
    "presion_min (hPa)", "presion_med (hPa)", "viento_vel_max (km/h)", "viento_vel_med (km/h)"
]]
# # limpiar np_100
    # df["np_100"] = df["np_100"].astype("Int64")
    # # limpiar np_300
    # df["np_300"] = df["np_300"].astype("Int64")

df["dias_precip>10mm"] = np.round(df["dias_precip>10mm"]).astype("Int64")
df["dias_precip>30mm"] = np.round(df["dias_precip>30mm"]).astype("Int64")

df.to_parquet("../data/processed/datos_meteo_Valencia_91-25(month).parquet")

print(df.info())

display(df)

<class 'pandas.DataFrame'>
PeriodIndex: 420 entries, 1991-01 to 2025-12
Freq: M
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   temp_med_mar (Cº)      420 non-null    float64
 1   temp_med_mes (Cº)      420 non-null    float64
 2   temp_med_max (Cº)      420 non-null    float64
 3   temp_med_min (Cº)      420 non-null    float64
 4   temp_abs_max (Cº)      420 non-null    float64
 5   temp_abs_min (Cº)      420 non-null    float64
 6   dias_temp>30           420 non-null    int64  
 7   total_precip_mes (mm)  420 non-null    float64
 8   precip_max (mm)        420 non-null    float64
 9   dias_precip>10mm       420 non-null    Int64  
 10  dias_precip>30mm       420 non-null    Int64  
 11  dias_lluvia            420 non-null    int64  
 12  dias_granizo           420 non-null    int64  
 13  dias_tormenta          420 non-null    int64  
 14  humedad_med (%)        420 non-null    float64
 15

,temp_med_mar (Cº),temp_med_mes (Cº),temp_med_max (Cº),temp_med_min (Cº),temp_abs_max (Cº),temp_abs_min (Cº),dias_temp>30,total_precip_mes (mm),precip_max (mm),dias_precip>10mm,dias_precip>30mm,dias_lluvia,dias_granizo,dias_tormenta,humedad_med (%),presion_max (hPa),presion_min (hPa),presion_med (hPa),viento_vel_max (km/h),viento_vel_med (km/h)
1991-01,13.6,11.1,15.3,6.8,22.5,1.2,0,104.2,32.4,3,2,7,1,1,74.580645,1030.4,1013.4,1022.179301,45.4,20.441801
1991-02,12.5,10.9,15.1,6.6,21.5,0.4,0,67.7,29.5,2,0,7,0,1,74.302083,1028.3,997.6,1014.741815,49.9,19.540625
1991-03,13.4,14.5,18.9,10.1,26.6,6.0,0,82.7,46.0,2,1,15,0,0,76.641129,1025.8,984.3,1009.190591,40.3,16.776747
1991-04,14.5,14.7,19.5,9.8,24.5,5.6,0,37.5,14.9,2,0,10,0,1,70.225000,1023.7,999.2,1013.787639,52.7,16.190417
1991-05,16.5,16.2,21.3,11.0,27.5,6.0,0,21.3,5.4,0,0,8,0,1,66.404570,1030.7,1004.2,1015.025538,30.9,12.288844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,27.9,28.3,32.3,24.2,38.2,22.5,30,2.4,2.4,0,0,3,0,1,59.756720,1019.5,1002.2,1012.525134,31.0,12.263172
2025-09,25.8,24.9,28.7,21.0,32.8,16.7,11,121.5,96.8,2,1,6,0,3,65.148611,1022.1,1004.8,1014.981528,52.6,14.542361
2025-10,22.9,21.3,25.2,17.3,30.0,11.2,2,178.3,65.6,5,2,10,0,3,75.264785,1024.0,1004.7,1015.425941,51.6,15.628495
2025-11,19.4,16.3,21.0,11.6,27.8,6.2,0,9.9,6.8,0,0,7,0,1,57.848611,1025.1,1005.4,1014.623611,50.2,18.703611
